# 04 — Single-line-to-ground fault

**Goal:** run the bundled SLG fault demonstrator, inspect the faulted network state, read the current, then verify the saved evidence.

**Teaching inputs:** bus 675, phase A, rf = 0.001 ohm.

**Prediction:** the fault current should be positive and finite for the declared source and feeder model.

Run the numbered cells in order. The direct OpenDSS section at the end is optional.

In [1]:
#@title 1. Setup — run once
from hashlib import sha256
from urllib.request import urlopen

_bootstrap_url = "https://raw.githubusercontent.com/sarutesri/cept-studio-edu/main/public/notebooks/_lesson.py"
_bootstrap = urlopen(_bootstrap_url, timeout=60).read()
if sha256(_bootstrap).hexdigest() != "aa5805c306987984ba7ee64d57763db1938cb06052cf80e0f8f26fa84efba30d":
    raise ValueError("Lesson helper hash mismatch")
exec(compile(_bootstrap, "cept-lesson", "exec"), globals())


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version


cept-power-studio 0.2.0.dev0
Lesson helpers ready. Stage cells below run the same CEPT commands as a normal terminal.


In [2]:
#@title 2. Inputs — fault location and impedance
RUN_DIR = WORKSPACE / "runs" / "04-fault-study"
FAULT_BUS = "675"
FAULT_PHASE = 1
FAULT_RESISTANCE_OHM = 0.001
table(
    ["declared input", "value", "unit"],
    [
        ("network", "IEEE 13-node feeder", "text"),
        ("fault bus", FAULT_BUS, "bus"),
        ("fault phase", "A", "phase"),
        ("fault resistance", FAULT_RESISTANCE_OHM, "ohm"),
    ],
)

declared input    value                unit
----------------  -------------------  -----
network           IEEE 13-node feeder  text
fault bus         675                  bus
fault phase       A                    phase
fault resistance  0.001                ohm


In [3]:
# 3. Run study — same CEPT CLI as a normal terminal
!cept study demo fault \
    --network ieee13 \
    --out runs/04-fault-study \
    --force \
    --format text

CEPT study result: FINISHED
----------------------------
Result             Finished the fault study and saved the evidence
Saved run          runs\04-fault-study
Case fingerprint   44d3766e5e8c (matches the case you ran)

What this means
  The study completed and saved solver-backed evidence.
  It does NOT approve a real project or field installation.

Next
  cept study verify runs\04-fault-study --format text


In [4]:
#@title 4. Explore — SLD and bus status
RUN_DIR = WORKSPACE / "runs" / "04-fault-study"
display_sld(RUN_DIR)

Bus,Phase A,Phase B,Phase C,Status
611,—,—,1.2235 pu @ 132.01°,OVER
632,0.5915 pu @ -2.77°,1.1656 pu @ -131.51°,1.1292 pu @ 126.18°,OUT
633,0.5904 pu @ -2.79°,1.1627 pu @ -131.54°,1.1267 pu @ 126.13°,OUT
634,0.5763 pu @ -3.48°,1.1432 pu @ -131.97°,1.1078 pu @ 125.70°,OUT
645,—,1.1562 pu @ -131.70°,1.1272 pu @ 126.21°,OVER
646,—,1.1547 pu @ -131.78°,1.1253 pu @ 126.26°,OVER
650,0.9988 pu @ -0.02°,0.9998 pu @ -119.99°,0.9998 pu @ 119.97°,OK
652,0.1054 pu @ -37.66°,—,—,UNDER
670,0.4243 pu @ -5.48°,1.2218 pu @ -134.46°,1.1593 pu @ 128.24°,OUT
671,0.1058 pu @ -38.18°,1.3467 pu @ -139.55°,1.2279 pu @ 132.28°,OUT


In [5]:
#@title 5. Engineering result — fault current
results = read(RUN_DIR / "results.json")
fault = results["fault"]
table(
    ["quantity", "value", "unit"],
    [
        ("fault bus", fault["bus"], "bus"),
        ("fault type", fault["fault_type"], "text"),
        ("fault resistance", fault["rf_ohm"], "ohm"),
        ("fault phases", fault["phases"], "phase"),
        ("total fault current", fault["total_fault_current_a"], "A"),
    ],
)
assert fault["bus"].lower() == FAULT_BUS.lower()
assert fault["phases"] == [FAULT_PHASE]

quantity             value   unit
-------------------  ------  -----
fault bus            675     bus
fault type           slg     text
fault resistance     0.001   ohm
fault phases         [1]     phase
total fault current  2950.7  A


In [ ]:
#@title 6. Plot — fault-current agreement and difference vs tolerance
import matplotlib.pyplot as plt

cept_current_a = float(fault["total_fault_current_a"])
vals = [float(direct_current_a), cept_current_a]
diff_a = abs(vals[0] - vals[1])
mean_a = sum(vals) / 2.0
rel_pct = (diff_a / mean_a * 100.0) if mean_a else 0.0

fig, (axTop, axBot) = plt.subplots(2, 1, figsize=(7, 5), gridspec_kw={"height_ratios": [3, 1.3]})
bars = axTop.bar(["Direct OpenDSS", "CEPT"], vals, color=["#1f5b4d", "#d5654e"], edgecolor="white")
axTop.set_title(f"SLG fault at bus {FAULT_BUS}-A: solver agreement")
axTop.set_ylabel("Fault current (A)")
axTop.set_ylim(0, max(vals) * 1.22)
for bar, v in zip(bars, vals):
    axTop.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(vals) * 0.02,
               f"{v:,.1f} A", ha="center", va="bottom", fontsize=10)
axTop.text(0.5, 0.88, f"\u0394 = {diff_a:.2f} A ({rel_pct:.3f}%)  ·  limit 1.0 A",
         transform=axTop.transAxes, ha="center", fontsize=9, color="#334155",
         bbox=dict(boxstyle="round,pad=0.3", fc="#f1f5f9", ec="#cbd5e1"))
axTop.grid(axis="y", alpha=0.25)

axBot.bar(["|Direct \u2212 CEPT|"], [diff_a], color="#334155", width=0.4)
axBot.axhline(1.0, color="#d5654e", linestyle="--", linewidth=1.5, label="Tolerance 1.0 A")
axBot.set_ylabel("\u0394 (A)")
axBot.set_ylim(0, max(diff_a * 2.2, 1.6))
axBot.legend(frameon=False, fontsize=8); axBot.grid(axis="y", alpha=0.25)
fig.tight_layout()
from IPython.display import display
display(fig)
plt.close(fig)
print(f"Agreement: \u0394 = {diff_a:.2f} A against a 1.0 A limit — worth one glance, not one decimal.")


In [6]:
# 6. Verify — check this exact saved run
!cept study verify runs/04-fault-study --format text

CEPT study check: PASSED
----------------------------
Study              Fault study (OpenDSS)
Case fingerprint   44d3766e5e8c (matches the case you ran)

Checked   3 groups, 13 checks, all passed
  [PASS] Case identity (4 checks)
  [PASS] Solver result (2 checks)
  [PASS] Saved evidence (7 checks)

What this means
  The saved result matches its Case, solver run, and saved evidence.
  It does NOT approve a real project or field installation.

Saved evidence     public-verification.json

For the full check list
  cept study verify . --format json


## 7. Interpret

The reported current belongs to the declared source, feeder, fault location, phase, and resistance.

**What this proves:** the solver-backed fault result was persisted and its workflow evidence can be verified.

**What this does not prove:** breaker interrupting duty, relay settings, arc-flash results, or field short-circuit acceptance.

**Try next:** predict what should happen if fault resistance increases, then compare with a deliberately changed direct-solver calculation.

## Optional — direct OpenDSS comparison

The cells below solve the same declared fault directly in OpenDSS and compare the returned fault-element current with the persisted CEPT result.

In [7]:
#@title Under the hood - direct OpenDSS fault solve (optional)
MASTER_DSS = ieee13_master()
import opendssdirect as dss

dss.Basic.ClearAll()
dss.Basic.DataPath(str(MASTER_DSS.parent))
dss.Text.Command(f'Redirect "{MASTER_DSS}"')
dss.Text.Command(f'New Fault.lesson_fault Bus1={FAULT_BUS}.{FAULT_PHASE} phases=1 r={FAULT_RESISTANCE_OHM}')
dss.Text.Command('Solve')
assert dss.Solution.Converged()
os.chdir(WORKSPACE)
print("Direct OpenDSS fault solve finished: the faulted feeder converged.")


Direct OpenDSS fault solve finished: the faulted feeder converged.


In [8]:
#@title Under the hood — direct fault readback (optional)
dss.Circuit.SetActiveElement('Fault.lesson_fault')
currents = dss.CktElement.Currents()
direct_current_a = abs(complex(currents[0], currents[1]))
table(['source', 'bus', 'phase', 'fault resistance', 'current', 'units'], [('direct OpenDSS', FAULT_BUS, FAULT_PHASE, FAULT_RESISTANCE_OHM, direct_current_a, 'ohm / A')])
assert direct_current_a > 0

source          bus  phase  fault resistance  current            units
--------------  ---  -----  ----------------  -----------------  -------
direct OpenDSS  675  1      0.001             2950.698148621237  ohm / A


In [10]:
#@title Compare solver outputs (optional details)
RUN_DIR = WORKSPACE / "runs" / "04-fault-study"
results = read(RUN_DIR / "results.json")
verify_summary = read(RUN_DIR / "public-verification.json")
fault = results["fault"]
cept_current_a = float(fault["total_fault_current_a"])
fault_abs_diff_a = abs(direct_current_a - cept_current_a)
table(
    ["source", "total fault current", "unit"],
    [("direct OpenDSS", direct_current_a, "A"), ("CEPT results.json", cept_current_a, "A")],
)
print()
print("Direct OpenDSS and CEPT fault currents agree")
print("-------------------------------------------")
print(f"Result        {'PASSED' if verify_summary['passed'] else 'Needs attention'}")
print(f"Fault current is about {cept_current_a:.1f} A")
print(f"The two solvers differ by {fault_abs_diff_a:.2f} A (limit 1.0 A)")
assert verify_summary["status"] == "PASS"
assert verify_summary["passed"] is True
assert fault["bus"].lower() == FAULT_BUS.lower() and fault["fault_type"] == "slg"
assert fault["phases"] == [FAULT_PHASE]
assert fault_abs_diff_a < 1.0


source             total fault current  unit
-----------------  -------------------  ----
direct OpenDSS     2950.698148621237    A
CEPT results.json  2950.7               A

Direct OpenDSS and CEPT fault currents agree
-------------------------------------------
Result        PASSED
Fault current is about 2950.7 A
The two solvers differ by 0.00 A (limit 1.0 A)
